# Backend selection v2 — vLLM vs llama.cpp on Xeon Max (cache mode)

Analysis notebook for `backend_comparison/`. All numbers are read from `processed_results/` (built by
`scripts/analysis/process.py` from the raw results); all figures are produced by `scripts/analysis/figures.py`
and saved in `plots/` with their data. The written conclusions are in `report/backend_selection_report.md`.

Arms: **vLLM BF16** and **llama.cpp BF16** form the precision-matched pair (same weights, same precision).
**llama.cpp Q8_0** is llama.cpp's own best configuration on this CPU and differs in precision; it is shown, never
divided into the BF16 numbers.

In [1]:
import sys
from pathlib import Path
import pandas as pd
import plotly.io as pio
from IPython.display import display, HTML
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT / 'scripts' / 'analysis'))
import figures as F
OUT = ROOT / 'plots'
P = ROOT / 'processed_results'
stream = pd.read_csv(P / 'stream_calibration.csv')
off = pd.read_csv(P / 'offline.csv')
pts = pd.read_csv(P / 'online_points.csv')
reps = pd.read_csv(P / 'online_reps.csv')
pts_main = pts[pts.campaign == 'main']
# (re)build every figure from the processed tables
import subprocess; subprocess.run([sys.executable, str(ROOT / 'scripts' / 'analysis' / 'figures.py')], check=True)

def show(name):
    p = OUT / f'{name}.html'
    if p.exists():
        display(HTML(p.read_text()))
    else:
        print(f'(figure {name} not available: no data)')

figures in /Users/lucian/University/llm-backend-analysis/backend_comparison/plots


## Validity checks

In [2]:
cols = ['model','arm','workload','concurrency','n_reps','tokens_ok','manifest_consistent','failed_requests',
        'total_tok_s_cv','client_cpu_frac_mean','hosts']
display(pts_main[cols].sort_values(['model','workload','arm','concurrency']))

,model,arm,workload,concurrency,n_reps,tokens_ok,manifest_consistent,failed_requests,total_tok_s_cv,client_cpu_frac_mean,hosts
0,llama31_8b,llamacpp-bf16,balanced,1,3,True,True,0,0.000128,0.003952,cresco8-hbm05
1,llama31_8b,llamacpp-bf16,balanced,2,3,True,True,0,0.000766,0.005281,cresco8-hbm05
2,llama31_8b,llamacpp-bf16,balanced,4,3,True,True,0,0.000743,0.007175,cresco8-hbm05
3,llama31_8b,llamacpp-bf16,balanced,8,3,True,True,0,0.000793,0.009094,cresco8-hbm05
4,llama31_8b,llamacpp-bf16,balanced,16,3,True,True,0,0.000648,0.010616,cresco8-hbm05
...,...,...,...,...,...,...,...,...,...,...,...
140,llama32_1b,vllm-bf16,prefill,4,3,True,True,0,0.028958,0.027236,cresco8-hbm01
141,llama32_1b,vllm-bf16,prefill,8,3,True,True,0,0.017767,0.033649,cresco8-hbm01
142,llama32_1b,vllm-bf16,prefill,16,3,True,True,0,0.038494,0.037496,cresco8-hbm01
143,llama32_1b,vllm-bf16,prefill,32,3,True,True,0,0.009438,0.036330,cresco8-hbm01


**Validity:**
- 0 failed requests out of 27,672 in the main campaign.
- Prompt and completion token counts are exact for every request on both servers.
- Each point's prompt manifest is identical across its 3 repetitions.
- The median coefficient of variation of throughput is 0.17% (max 6.0%).
- Client CPU ≤ 0.30 of one core.
- Compute-core busy during idle baselines ≤ 0.85%.

## Experiment 0 — sustainable bandwidth and the PMU proxy

**Plotted:** STREAM triad bandwidth on the server layout (socket 0, 55/56 threads, cache mode) for working
sets from 4 to 96 GiB (left), and the ratio between what the core PMU counters report and STREAM's modeled
traffic (right).
**Why it matters:** every bandwidth number for the inference servers comes from the same core counters. If the
counters reproduce STREAM's known traffic, application bandwidth can be normalized by STREAM measured *with the
same instrument*, and the normalization does not depend on the theoretical HBM peak.

In [3]:
F.stream_figure(stream, OUT); show('stream_calibration')

**What can be observed.** STREAM on the server layout sustains ~505–519 GB/s for working sets ≤ 8 GiB. It falls to
484/383/302/252/191 GB/s at 16/32/48/64/96 GiB as the direct-mapped 64 GiB HBM cache is shared and then
exceeded. The core-PMU reads + writes equal 0.97–1.00× STREAM's modeled traffic at every size, with no
multiplexing (running fraction 1.00). This instrument is used for all server traffic.

**What cannot be concluded from this plot alone.** The counters see memory-side traffic from the cores. In cache mode they cannot split HBM-cache hits from DDR
traffic, and they do not see the HBM cache's own fills and evictions toward DDR. The ratio is a consistency check
against STREAM's model, not a calibration against device-level CAS counts (not readable at
`perf_event_paranoid=2`).

## Experiment A — offline prompt processing

**Plotted:** prompt-processing tokens/s from each engine's own benchmark tool, at batch 1 (`llama-bench pp`
vs `vllm bench latency --output-len 1`) and at fixed batch 8 and 32 (`llama-batched-bench` vs `vllm bench latency`).
**Why it matters:** prefill is compute-bound; it shows how much of each engine's matrix work reaches AMX.

In [4]:
F.offline_bars(off, 'prefill', 'f01_offline_prefill', 'Offline prompt-processing throughput', OUT); show('f01_offline_prefill_llama31_8b')

**What can be observed.** vLLM prefills 5.4× faster than llama.cpp BF16 (1,236 vs 229 tok/s at batch 1, 8B) and 2.9× faster than
Q8_0. Neither engine's prefill rate grows much with batch, since prefill is already a large matrix product.
The online C=1 TTFTs imply the same prefill rates (1,219 and 219 tok/s).

**What cannot be concluded from this plot alone.** The tools differ in scope. vLLM's number includes scheduling, input preparation and sampling of one token;
llama-bench times forward passes only. Small differences are not engine differences.

## Experiment A — offline generation

**Plotted:** generation tokens/s: `llama-bench tg` with an ISL-token context vs vLLM's
(OSL−1)/(full − prefill) at batch 1, and both engines' batched decode at batch 8 and 32.
**Why it matters:** single-stream decode is limited by reading the weights once per token; batched decode shows
whether an engine amortizes that read over several sequences.

In [5]:
F.offline_bars(off, 'decode', 'f02_offline_decode', 'Offline generation throughput', OUT); show('f02_offline_decode_llama31_8b')

**What can be observed.** At batch 1, llama.cpp BF16 decodes 18% faster than vLLM (21.7 vs 18.4 tok/s) and Q8_0 69% faster. At
batch 32 the order reverses: vLLM 421, llama.cpp BF16 139, Q8_0 229 tok/s (3.0× and 1.8×).

**What cannot be concluded from this plot alone.** vLLM's decode rate is derived from two separate runs (median prefill subtracted from each full run), so its
error bar understates the uncertainty. llama.cpp Q8_0 reads half the weight bytes of BF16; its batch-1 advantage
is partly precision, not engine.

## Experiment B — online throughput vs concurrency

**Plotted:** steady-state output tokens/s (and total tokens/s) of each server under a closed loop with C
requests in flight, for each workload shape. Error bars: ±1 SD over 3 repetitions.
**Why it matters:** this is the primary selection criterion — how much aggregate work a server delivers as
concurrency rises.

In [6]:
for m in ('f03_output_throughput', 'f04_total_throughput'): show(f'{m}_llama31_8b')

**What can be observed.** vLLM's throughput keeps rising up to C=64–128 on short prompts (849–857 tok/s, 23–37× its C=1). llama.cpp
flattens at C=16–32 (156–158 BF16, 251–257 Q8_0, 4–7× its C=1). On long prompts llama.cpp stops gaining
at C=2–4 (≤1.2×) while vLLM reaches 1,013–1,175 tok/s. At C=1 llama.cpp BF16 is 12–16% ahead on the
decode-dominated shapes.

**What cannot be concluded from this plot alone.** Throughput alone hides latency: a higher curve may be bought with unusable per-request latency (next plots).
Different precisions (Q8_0) are not engine comparisons.

## Scaling efficiency

**Plotted:** speedup over C=1 divided by C. 1.0 would be perfect linear scaling.
**Why it matters:** it shows how much of each additional concurrent request turns into extra throughput.

In [7]:
show('f12_scaling_efficiency_llama31_8b')

**What can be observed.** vLLM keeps ≥ 0.87 scaling efficiency up to C=8, 0.35–0.53 at C=64 and 0.18–0.29 at C=128. llama.cpp BF16
is at 0.37–0.50 at C=8 and 0.06–0.09 at C=64. The ratio vLLM ÷ llama.cpp (BF16) is 0.86–0.89× at C=1, crosses 1 at C≈2 and reaches
4.9–6.0× at the highest common C.

**What cannot be concluded from this plot alone.** Efficiency is relative to each backend's own C=1, so a backend that is slow at C=1 can look efficient. Read it
together with absolute throughput.

## Latency under load: TTFT, TPOT, end-to-end

**Plotted:** median TTFT, TPOT and E2E latency of steady-state requests (log scale).
**Why it matters:** a serving backend must keep per-request latency usable while throughput grows.

In [8]:
for m in ('f05_ttft', 'f06_tpot', 'f07_e2e'): show(f'{m}_llama31_8b')

**What can be observed.** At C=1 llama.cpp has the lower TPOT (−16 to −18% BF16, −40% Q8_0), and vLLM the lower TTFT (4–8×). From
C≈4 vLLM's TPOT grows slowly: on 512→512, 53 → 97 → 148 ms at C = 1/32/64. llama.cpp BF16's TPOT
grows close to linearly with C: 43 → 397 → 804 ms. At C=32 the pooled TPOT p99 is 98 ms for vLLM vs
398 ms for llama.cpp BF16.

**What cannot be concluded from this plot alone.** Medians are per repetition and averaged; pooled p95/p99 are in `online_points.csv` with their sample counts.
p99 is reported only where at least 100 steady requests exist.

## Concurrency → batching

**Plotted:** tokens processed per forward step, from each server's own counters (vLLM
`iteration_tokens_total`, llama.cpp `n_decode_total` and token counters), and AMX busy fraction.
**Why it matters:** it is the link between "more concurrent requests" and "more work per weight read".

In [9]:
for m in ('f14_tokens_per_step', 'f15_amx_busy', 'f16_cpu_util'): show(f'{m}_llama31_8b')

**What can be observed.** Both servers batch: tokens per forward step ≈ C (C=64, 512→512: llama.cpp 109, vLLM 118). The difference
is step speed. From batch ~2 to ~118, vLLM's step time grows 2.7×, llama.cpp's 16×. vLLM's AMX busy
fraction rises 0.4% → 11% (18% on 4096→128) with all compute cores busy. llama.cpp BF16 never uses AMX,
and its core occupancy drops to 87% (Q8_0: 81%) at C=64.

**What cannot be concluded from this plot alone.** Tokens per step mixes prefill chunks and decode tokens. It shows batching took place, not how efficiently each
step used the hardware.

## Memory traffic vs concurrency

**Plotted:** memory read and write traffic of the server process tree (core PMU, beyond L3), and the same
traffic divided by STREAM's sustainable traffic measured with the same counters.
**Why it matters:** this is the question the thesis needs answered: does more concurrency produce more memory
traffic, and how close does it get to the machine's sustainable bandwidth?

In [10]:
for m in ('f08_mem_read_bw', 'f09b_mem_write_bw', 'f10_bw_utilization'): show(f'{m}_llama31_8b')

**What can be observed.** Memory traffic is highest at C=1 and falls as C rises, for every backend and workload. Single-stream decode
reads all weights per token: llama.cpp BF16 moves 319–325 GB/s (63–65% of sustainable), vLLM 278–282 GB/s
(55–56%). Batching amortizes that read, so at C=64 vLLM moves 176–197 GB/s and llama.cpp 45–48 GB/s. In long-KV
decode (4096→1024) KV reads grow with C (vLLM model: 10 → 95 GB/s) and dominate from C≈32, yet the total
still falls (263 → 154 GB/s) because each step becomes slower than the memory system.
The first-principles model (steps/s × streamed weight bytes + decode tok/s × context × KV bytes) is a lower
bound; wherever decode dominates the measurement is 0.96–1.18× of it for vLLM (median 1.01) and 0.96–1.52× for
llama.cpp (median 1.05–1.15).

**What cannot be concluded from this plot alone.** In cache mode the traffic is HBM-cache hits plus DDR, not separable here. A bandwidth plateau alone does not
prove bandwidth saturation; it must coincide with a throughput plateau and with compute not being the limit
(see the saturation table).

## The selection figure: throughput and memory traffic together

**Plotted:** for each workload, aggregate output throughput (top) and memory traffic (bottom) against
concurrency, on shared x axes. The dotted line is STREAM's sustainable traffic.
**Why it matters:** it answers which backend turns additional concurrent requests into more throughput *and*
more memory-system load.

In [11]:
show('key_throughput_and_bandwidth_llama31_8b'); show('throughput_vs_bandwidth_llama31_8b')

**What can be observed.** vLLM is the backend whose throughput keeps responding to concurrency. Its memory traffic stays at 28–56%
of sustainable across C=1…128, while llama.cpp's falls to 13–15% at C=32 (where its throughput stops
growing) and 9% at C=64. Neither reaches the STREAM line: concurrency moves both backends
towards compute-bound operation, not towards HBM saturation.

**What cannot be concluded from this plot alone.** Separate panels, not a dual axis: the two quantities have different units and the eye must not compare their
heights.

## Cross-check with the 1B model

**Plotted:** the same key figure for Llama-3.2-1B.
**Why it matters:** it tests whether the 8B behaviour is a property of the backend or of the 8B model.

In [12]:
show('key_throughput_and_bandwidth_llama32_1b'); show('f06_tpot_llama32_1b')

**What can be observed.** The 1B model reproduces every pattern:
- llama.cpp wins C=1 (TPOT 9.0 vs 12.0 ms);
- vLLM scales to C=256 (4,446–5,083 tok/s, 31–43× its C=1) while llama.cpp plateaus at C=16–32 (≤ 638 BF16, ≤ 912 Q8_0);
- memory traffic falls with C for both.

So the behaviour belongs to the backends, not to the 8B model.

**What cannot be concluded from this plot alone.** The 1B model has a 6.5× smaller weight read per step and 4× less KV per token, so its bandwidth demand per
token is much lower; absolute bandwidth numbers are not comparable across models.

## Saturation analysis

For every doubling of C: throughput gain, memory-traffic gain and median-TPOT increase. Saturation is claimed only
where throughput and memory traffic both plateau while latency keeps rising.

In [13]:
sat = pts_main[['model','arm','workload','concurrency','total_tok_s_mean','gain_vs_prev','mem_total_gbs_mean',
                'bw_gain_vs_prev','tpot_increase_vs_prev','bw_util_vs_stream_max_mean','cpu_busy_compute_mean_mean',
                'amx_busy_frac_mean','tokens_per_step_mean']]
display(sat.sort_values(['model','workload','arm','concurrency']).round(3))

,model,arm,workload,concurrency,total_tok_s_mean,gain_vs_prev,mem_total_gbs_mean,bw_gain_vs_prev,tpot_increase_vs_prev,bw_util_vs_stream_max_mean,cpu_busy_compute_mean_mean,amx_busy_frac_mean,tokens_per_step_mean
0,llama31_8b,llamacpp-bf16,balanced,1,41.917,NaN,318.606,NaN,NaN,0.634,0.996,0.000,1.880
1,llama31_8b,llamacpp-bf16,balanced,2,66.005,0.575,252.331,-0.208,0.342,0.502,0.997,0.000,3.579
2,llama31_8b,llamacpp-bf16,balanced,4,102.529,0.553,199.607,-0.209,0.330,0.397,0.997,0.000,7.728
3,llama31_8b,llamacpp-bf16,balanced,8,124.034,0.210,139.356,-0.302,0.613,0.277,0.991,0.000,13.810
4,llama31_8b,llamacpp-bf16,balanced,16,143.336,0.156,94.701,-0.320,0.743,0.189,0.957,0.000,27.238
...,...,...,...,...,...,...,...,...,...,...,...,...,...
140,llama32_1b,vllm-bf16,prefill,4,5014.190,0.378,146.816,-0.120,0.469,0.292,0.998,0.086,119.272
141,llama32_1b,vllm-bf16,prefill,8,6408.684,0.278,133.709,-0.089,0.621,0.266,0.997,0.110,247.148
142,llama32_1b,vllm-bf16,prefill,16,7197.037,0.123,126.020,-0.058,0.730,0.251,0.992,0.132,476.677
143,llama32_1b,vllm-bf16,prefill,32,7000.146,-0.027,113.595,-0.099,0.870,0.226,0.990,0.136,844.448


No configuration saturates memory bandwidth: the largest share of sustainable is 65%, reached at C=1.
- **vLLM:** saturates by compute. Throughput plateaus at C=64–128 (+2%/+9% for the last doubling), TPOT doubles, cores ≥ 98.6% busy, AMX busy 10–18%, memory traffic falling.
- **llama.cpp:** plateaus earlier with lower core occupancy and very low memory traffic, i.e. limited by its per-step execution cost.

## Fact / observation / interpretation / conclusion

- **FACT:** vLLM BF16 = 4.9–6.0× llama.cpp BF16 throughput at the highest common C. llama.cpp BF16 = 1.12–1.16× vLLM at C=1. Memory traffic falls with C for both, with maxima of 65% (llama.cpp, C=1) and 56% (vLLM, C=1) of sustainable.
- **OBSERVATION:** both batch equally. vLLM's step time grows 2.7× from batch 2 to ~118, llama.cpp's 16×.
- **INTERPRETATION:** batching amortizes the weight read, so serving becomes compute-bound. vLLM's AMX coverage lets its compute scale; llama.cpp's per-token cost at large batch caps it. The attention attribution comes from the earlier study and was not re-measured here.
- **CONCLUSION:** vLLM is the platform for the concurrent-serving thesis. HBM pressure should be studied at low-batch decode and in long-KV decode, not by raising concurrency on short contexts.